In [14]:
import os
import torch
from bioio import BioImage, Scale
import bioio_tifffile
import numpy as np
import stackview

print (f"Using cuda device: {torch.cuda.is_available()}")

Using cuda device: True


In [15]:
import vistiq
from vistiq.core import ArrayIteratorConfig
from vistiq.io import ImageLoaderConfig, ImageLoader
from vistiq.preprocess import DoGConfig, DoG
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig
from vistiq.segment.select import RegionFilter, RegionFilterConfig, RangeFilterConfig, RangeFilter
from vistiq.segment.label import MicroSAMSegmenterConfig, MicroSAMSegmenter
from vistiq.segment.postprocess import WatershedConfig, Watershed

# Load image

In [54]:
path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"

ilc = ImageLoaderConfig(squeeze=True, rename_channel={"Channel:0:0": "Dpn"}, substack="Z:65-70")
img, metadata = ImageLoader(ilc).run(path)
metadata

2026-05-01 10:13:29,329 - INFO - Loading image from: /standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif
2026-05-01 10:13:29,511 - INFO - Scenes found: ('Image:0',)
2026-05-01 10:13:29,512 - INFO - Applying substack={'Z': slice(64, 70, None)}
2026-05-01 10:13:29,535 - INFO - Loaded image: /standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif scene=default -> shape=(6, 512, 512) dtype=uint8, channel_names=['Dpn']
2026-05-01 10:13:29,535 - INFO - Loaded image with shape: (6, 512, 512), dtype: uint8
/home/khs3z/.conda/envs/vistiq-env-gpu/lib/python3.12/logging/__init__.py:1962: UserWarning: Logger 'prefect.task_runs' attempted to send logs to the API without a flow run id. The API log handler can only send logs within flow run contexts unless the flow run id is manually provided. Set PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW=ignore to suppress this warning.
  self.logger.log(level, msg, *args, **kwarg

{'scene_index': None,
 'dim_order': 'ZYX',
 'axes': ['Z', 'Y', 'X'],
 'channel_names': ['Dpn'],
 'channel_axis': None,
 'shape': (6, 512, 512),
 'dims': <Dimensions [Z: 6, Y: 512, X: 512]>,
 'pixel_unit': 'um',
 'scale': Scale(T=None, C=None, Z=0.9999285714285713, Y=0.23288238764065222, X=0.23288238764065222),
 'physical_pixel_sizes': PhysicalPixelSizes(Z=0.9999285714285713, Y=0.23288238764065222, X=0.23288238764065222)}

# Set embedding path

In [55]:
z_start = 0
z_end = metadata["dims"].Z
embedding_dir = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"
embedding_paths = [os.path.join(embedding_dir, ch_label, f"{z_start}-{z_end}") for ch_label in metadata["channel_names"]]

# Preprocess

In [56]:
dogc = DoGConfig(
    sigma_low=1.0, 
    sigma_high=12.0
)
dimg,_ = DoG(dogc).run(img)

2026-05-01 10:13:29,561 - INFO - Running preprocessor DoG, on stack of type uint8, True
2026-05-01 10:13:29,566 - INFO - Running DoG with config: classname='DoGConfig' package='vistiq.preprocess' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 tile_shape=None output_type='stack' output_shape=None squeeze=True split_axis=None split_channels=False rename_channel=None normalize=True dtype=None sigma_low=1.0 sigma_high=12.0 mode='reflect'
2026-05-01 10:13:29,567 - INFO - StackProcessor.run: received workers=1 (type: <class 'int'>)
2026-05-01 10:13:29,567 - INFO - Using Parallel with n_jobs=1 for 6 iterations
2026-05-01 10:13:29,568 - INFO - Processing slice, slice.shape=(512, 512)
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.0s
2026-05-01 10:13:29,597 - INFO - Processing slice, slice.shape=(512, 512)
2026-05-01 10:13:29,626 - INFO - Processing slice, slice.shape=(512, 512)
2026-0

# Define Region Analyzer

In [57]:
itc = ArrayIteratorConfig(slice_def=(-3,-2,-1))
rac = RegionAnalyzerConfig(
    properties=["volume", "cross_sectional_area", "area", "aspect_ratio", "bbox"], 
    iterator_config=itc, 
    output_type="dataframe"
)
ra = RegionAnalyzer(rac)

# Define Region Filter

In [58]:
rfc = RegionFilterConfig(filters=[
    RangeFilter(
        RangeFilterConfig(
            attribute="cross_sectional_area", 
            range=(15.0,2000.0)
        )
    ),
    RangeFilter(
        RangeFilterConfig(
            attribute="volume",
            range=(10,20000)
        )
    ),
])
rf = RegionFilter(rfc)

# Segment

In [59]:
segc = MicroSAMSegmenterConfig(
    embedding_path=embedding_paths[0], 
    do_regions=True, 
    region_analyzer=ra, 
    region_filter=rf
)
dmask, dlabels, dresults = MicroSAMSegmenter(segc).run(dimg, metadata=metadata)

2026-05-01 10:13:29,773 - INFO - Labeller not provided, using default Labeller with connectivity=1 and region_filter=None
2026-05-01 10:13:29,773 - INFO - Segmenter config: classname=None package=None version=None command_group=None thresholder=None binary_processor=None labeller=Labeller(classname='LabellerConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 tile_shape=None output_type='list' output_shape=None squeeze=True split_axis=None split_channels=False rename_channel=None connectivity=1 region_filter=None) region_analyzer=RegionAnalyzer(classname='RegionAnalyzerConfig' package='vistiq.segment.analysis' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 tile_shape=None output_type='dataframe' output_shape=None squeeze=True split_axis=None split_chann

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'numpy.ndarray'>


2026-05-01 10:13:34,018 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/069f4b50-c311-7c0a-8000-104cbf92e868/set_state "HTTP/1.1 201 Created"
2026-05-01 10:13:34,470 - INFO - Finished in state Completed()


In [60]:
stackview.slice(dlabels, continuous_update=True)

# Results

In [61]:
dresults.describe()

,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,aspect_ratio,cross_sectional_area,volume
count,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,26.000000,31.000000,31.000000
mean,1.387097,241.000000,254.483871,4.645161,270.258065,281.612903,0.500715,138.279552,102.822460
std,1.873987,141.939424,114.662366,1.226061,143.114632,117.636921,0.166908,75.073343,95.571198
min,0.000000,39.000000,0.000000,2.000000,63.000000,12.000000,0.159419,47.508007,12.906819
25%,0.000000,92.500000,160.000000,4.000000,115.000000,188.000000,0.396820,69.515393,27.277857
50%,0.000000,290.000000,273.000000,5.000000,320.000000,308.000000,0.486842,126.222254,75.922466
75%,2.000000,366.000000,342.500000,6.000000,398.500000,372.000000,0.649308,188.518293,135.819868
max,5.000000,449.000000,433.000000,6.000000,476.000000,452.000000,0.805817,334.651991,408.354404


In [62]:
from joblib import Parallel, delayed

def check_labels(labels, results, i):
    coords = np.argwhere(labels == i)
    top_left = results[results.index==i][["bbox-0", "bbox-1", "bbox-2"]].to_numpy().squeeze()
    bottom_right = results[results.index==i][["bbox-3", "bbox-4", "bbox-5"]].to_numpy().squeeze()
    return np.array_equal(top_left, coords.min(axis=0)) and np.array_equal(bottom_right, coords.max(axis=0)+1)

label_nums = np.unique(dlabels)
results = Parallel(n_jobs=-1, verbose=10, batch_size=4)(delayed(check_labels)(dlabels, dresults, i) for i in range(1, label_nums[-1]+1))
assert all(results)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done  20 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done  31 out of  31 | elapsed:    1.3s finished


# Postprocess Labels: Watershed

In [94]:
wsc = WatershedConfig(
    footprint=(3,15,15), 
    h=2.5, 
    min_distance=0.5, 
    compactness=10
)
wlabels = Watershed(wsc).run(dlabels, metadata)

2026-05-01 10:21:18,713 - INFO - metadata['scale']=Scale(T=None, C=None, Z=0.9999285714285713, Y=0.23288238764065222, X=0.23288238764065222), min_distance=0.5, min_dist_pixel=0
2026-05-01 10:21:18,713 - INFO - footprint=(3, 15, 15)
2026-05-01 10:21:18,737 - INFO - Segmenting mask with intensity value=1; segment values=[0 1], last_label=0
2026-05-01 10:21:18,748 - INFO - Segmenting mask with intensity value=2; segment values=[0 1], last_label=1
2026-05-01 10:21:18,759 - INFO - Segmenting mask with intensity value=3; segment values=[0 1], last_label=2
2026-05-01 10:21:18,768 - INFO - Segmenting mask with intensity value=4; segment values=[0 1], last_label=3
2026-05-01 10:21:18,779 - INFO - Segmenting mask with intensity value=5; segment values=[0 1], last_label=4
2026-05-01 10:21:18,788 - INFO - Segmenting mask with intensity value=6; segment values=[0 1], last_label=5
2026-05-01 10:21:18,800 - INFO - Segmenting mask with intensity value=7; segment values=[0 1 2], last_label=6
2026-05-01

In [95]:
import stackview
stackview.slice(np.concatenate([dlabels, wlabels], axis=-1), continuous_update=True)